# Chapter 5 — Subspaces, Image, Kernel, Basis, Dimension: SageMath Companion

Symbolic, exact-arithmetic counterpart to `code/python/05_subspaces.ipynb`. Where the Python notebook used NumPy floats, this notebook uses SageMath's `Matrix(QQ, ...)` — pivots, ranks, and bases are all exact, and we can use Sage's built-in `.column_space()`, `.right_kernel()`, and `.rref()` to cross-check every hand computation from the notes.

**To run this notebook:** open in [CoCalc](https://cocalc.com/) (no install needed), or locally with the SageMath kernel (`sage -n jupyter`).

**Kernel:** SageMath.

**Layout:**

1. Subspace axioms — symbolic verification for a plane through the origin
2. Image (column space) of a matrix — by RREF and by Sage built-in
3. Kernel (null space) — by RREF and by Sage built-in
4. Linear independence test and the rank-column pivot rule
5. Extracting a basis from a redundant spanning set
6. Rank–nullity theorem on symbolic matrices
7. The invertible-matrix theorem — checking all 10 equivalent conditions
8. Coordinates in a non-standard basis
9. Change-of-basis matrix
10. Polynomial subspace — preview of Chapter 6
11. **Exercises** with solutions

## 1. Subspace — verifying the axioms symbolically

Let `V = {(x, y, z) ∈ ℝ³ : x + 2y − z = 0}`. We verify all three axioms with Sage's symbolic algebra — no numerics needed.

In [ ]:
u1, u2, v1, v2, c = var('u1 u2 v1 v2 c')

# A point of V has the form (x, y, z) with z = x + 2y. Parametrize:
def in_V(x, y):
    return vector(SR, [x, y, x + 2*y])

# Axiom 1: zero belongs
zero = in_V(0, 0)
show('0 in V?', zero == vector(SR, [0, 0, 0]))

# Axiom 2: sum of two V-points is in V
u = in_V(u1, u2)
w = in_V(v1, v2)
s = u + w
# Does it satisfy x + 2y - z = 0?
residual = (s[0] + 2*s[1] - s[2]).simplify_full()
show('u + v satisfies x + 2y - z = 0?  Residual =', residual)

# Axiom 3: scalar multiple of a V-point is in V
cu = c * u
residual2 = (cu[0] + 2*cu[1] - cu[2]).simplify_full()
show('c·u satisfies x + 2y - z = 0?  Residual =', residual2)

## 2. Image (column space) — two ways

The matrix from `notes.md` §5.0.

In [ ]:
A = matrix(QQ, [[1, 2, 3],
                [2, 4, 6],
                [1, 1, 2]])
show('A ='); show(A)
show('RREF(A) ='); show(A.rref())
show('pivot columns =', A.pivots())
show('rank A =', A.rank())

In [ ]:
# Basis of im A: the ORIGINAL columns corresponding to pivot columns of RREF
pivots = A.pivots()
basis_im = A.matrix_from_columns(list(pivots))
show('Basis of im A (as columns) ='); show(basis_im)

# Cross-check with Sage's built-in column_space.
# Note: column_space().basis() returns a basis but with respect to its own row-echelon form.
show('Sage column space:')
show(A.column_space())

## 3. Kernel (null space) — two ways

First the hand recipe (set each free variable to 1 in turn), then Sage's `right_kernel()`.

In [ ]:
# Free columns = everything not in A.pivots()
n = A.ncols()
free = [c for c in range(n) if c not in A.pivots()]
Rref = A.rref()

basis_ker = []
for f in free:
    v = [0] * n
    v[f] = 1
    for i, p in enumerate(A.pivots()):
        v[p] = -Rref[i, f]
    basis_ker.append(vector(QQ, v))

show('Free columns:', free)
show('Basis of ker A =', basis_ker)

# Cross-check: Sage
show('Sage right_kernel(A):')
show(A.right_kernel())

In [ ]:
# Verify: A @ kernel_basis should be zero
for v in basis_ker:
    show('A ·', v, '=', A * v)

## 4. Linear independence test

Stack vectors as columns, check if `rank == n_vectors`. If not, the kernel of the stacked matrix exhibits the dependency.

In [ ]:
v1 = vector(QQ, [1, 1, 2])
v2 = vector(QQ, [2, 3, 5])
v3 = vector(QQ, [1, 0, 1])

M = column_matrix([v1, v2, v3])
show('M ='); show(M)
show('rank M =', M.rank())
show('#columns =', M.ncols())
show('independent?', M.rank() == M.ncols())

# If dependent, find a dependency
K = M.right_kernel()
show('kernel:'); show(K)
show('A dependency coefficient vector:', K.basis()[0])

## 5. Basis from a redundant spanning set

Exactly the worked example E6. Stack the `wᵢ`'s as columns; pivot columns of RREF point to a basis.

In [ ]:
W = column_matrix([
    vector(QQ, [1, 0, 1, 0]),   # w1
    vector(QQ, [2, 1, 2, 1]),   # w2
    vector(QQ, [1, 1, 1, 1]),   # w3
    vector(QQ, [3, 1, 3, 1])    # w4
])
show('W ='); show(W)
show('RREF(W) ='); show(W.rref())
show('pivot columns =', W.pivots())

basis_V = W.matrix_from_columns(list(W.pivots()))
show('Basis of V (from the original wᵢ):'); show(basis_V)
show('dim V =', W.rank())

## 6. Rank–nullity symbolically and on a random family

Pick random rational matrices of a given shape and rank, and verify `rank + nullity = n`.

In [ ]:
set_random_seed(0)
for trial in range(5):
    m = ZZ.random_element(3, 7)
    n = ZZ.random_element(3, 7)
    r_target = ZZ.random_element(1, min(m, n) + 1)
    U = random_matrix(QQ, m, r_target, num_bound=3, den_bound=1)
    V = random_matrix(QQ, r_target, n, num_bound=3, den_bound=1)
    A_rand = U * V
    rank = A_rand.rank()
    nullity = A_rand.right_kernel().dimension()
    ok = (rank + nullity == n)
    print(f'Trial {trial+1}: shape {A_rand.nrows()}x{A_rand.ncols()}, rank {rank}, nullity {nullity}, sum = {rank+nullity} (n = {n}) {"✓" if ok else "FAIL"}')

## 7. The invertible-matrix theorem — checking all 10 conditions

Take a random invertible `3 × 3` rational matrix and check that every condition from §5.8 holds.

In [ ]:
set_random_seed(42)
while True:
    A = random_matrix(QQ, 3, 3, num_bound=3)
    if A.det() != 0:
        break

show('A ='); show(A)

conditions = {
    '1. A is invertible (det != 0)': A.det() != 0,
    '2. RREF(A) = I': A.rref() == identity_matrix(QQ, 3),
    '3. rank A = n':   A.rank() == 3,
    '4. ker A = {0}':  A.right_kernel().dimension() == 0,
    '5. im A = R^n':   A.column_space().dimension() == 3,
    '6. columns independent': column_matrix(A.columns()).rank() == 3,
    '7. columns span R^3':    column_matrix(A.columns()).rank() == 3,
    '8. columns are a basis':  A.rank() == 3 and A.right_kernel().dimension() == 0,
    '9. Ax=b unique for every b': A.det() != 0,
    '10. Ax=0 only trivial':      A.right_kernel().dimension() == 0,
}

for k, v in conditions.items():
    print(f'  {k}: {v}')

## 8. Coordinates in a non-standard basis

From `notes.md` §5.9 and worked example E8.

In [ ]:
v1 = vector(QQ, [1, 0, 0])
v2 = vector(QQ, [1, 1, 0])
v3 = vector(QQ, [1, 1, 1])

S_B = column_matrix([v1, v2, v3])
show('S_B ='); show(S_B)
show('det(S_B) =', S_B.det(), '  (non-zero → basis)')

x = vector(QQ, [4, 3, 2])
coords_B = S_B.solve_right(x)
show('[x]_B =', coords_B)
show('rebuild: S_B · [x]_B =', S_B * coords_B)

## 9. Change-of-basis matrix

Convert `[x]_𝓑 = (2, 1)` to `[x]_𝓒` with the two bases from `notes.md` §5.10.2:
`𝓑 = ((1, 1), (1, -1))`, `𝓒 = ((1, 0), (1, 2))`.

In [ ]:
S_B = matrix(QQ, [[1, 1], [1, -1]])   # columns: (1,1), (1,-1)
S_C = matrix(QQ, [[1, 1], [0, 2]])    # columns: (1,0), (1,2)

P = S_C.inverse() * S_B    # P_{B -> C}
show('P_{B -> C} ='); show(P)

x_B = vector(QQ, [2, 1])
x_C = P * x_B
show('[x]_B =', x_B, '  ->  [x]_C =', x_C)

# Cross-check: rebuild x both ways
show('x from B:', S_B * x_B)
show('x from C:', S_C * x_C)

## 10. Polynomial subspace — preview of Chapter 6

`P₂` = polynomials `p(x) = a₀ + a₁ x + a₂ x²`. We can treat every polynomial as its coefficient vector `(a₀, a₁, a₂)`. Then "subspace of `P₂`" means "subspace of ℝ³" — same theory.

Exercise E10: `V = { p ∈ P₂ : p(1) = 0 }`. Show `V` is 2D and find a basis.

In [ ]:
R.<x> = PolynomialRing(QQ)

def coeffs(p, deg=2):
    lst = p.list()
    return vector(QQ, lst + [0]*(deg + 1 - len(lst)))

p1 = x - 1         # p1(1) = 0 ✓
p2 = x^2 - 1       # p2(1) = 0 ✓

show('p1 =', p1, '  coeffs =', coeffs(p1))
show('p2 =', p2, '  coeffs =', coeffs(p2))

# Stack as columns and check independence
B = column_matrix([coeffs(p1), coeffs(p2)])
show('Basis matrix ='); show(B)
show('rank =', B.rank(), '  → dim V = 2')

In [ ]:
# Any p with p(1) = 0 should be a linear combination of p1 and p2.
# Try: p(x) = 3x^2 + x - 4.  Check p(1) = 3 + 1 - 4 = 0.
p = 3*x^2 + x - 4
show('p(1) =', p.subs(x=1))

# Find its V-coordinates by solving B · (c1, c2) = coeffs(p)
c = B.solve_right(coeffs(p))
show('V-coordinates of p in basis (p1, p2):', c)
# Rebuild
reconstructed = c[0] * p1 + c[1] * p2
show('reconstructed =', reconstructed.expand(), '  (should match p)')

---

## 11. Exercises (with solutions)

Each exercise asks you to do a calculation you can double-check with Sage. Solutions appear below — but **do the work by hand first**!

### E1 — Check whether a given plane equation defines a subspace

Let `W = {(x, y, z) : 2x − y + 3z = 0}`. Verify all three subspace axioms symbolically, then show the plane has dimension 2 by finding a basis.

### E2 — Image and kernel of a specific matrix

For

```
         ⎡ 1   2   1   0 ⎤
   M  =  ⎢ 2   4   1   1 ⎥
         ⎣ 1   2   2  −1 ⎦
```

find bases for `im M` and `ker M`, state the rank and nullity, and verify rank–nullity.

### E3 — Coordinates

In ℝ³, let `𝓑 = ((1, 2, 0), (0, 1, 1), (1, 0, 1))`. Verify it's a basis. Then compute `[x]_𝓑` for `x = (3, 2, 1)`.

### E4 — Change of basis in ℝ²

Let `𝓑 = ((2, 1), (1, 3))` and `𝓒 = ((1, 0), (1, 1))`. Build the matrix `P_{𝓒 → 𝓑}` that converts 𝓒-coordinates to 𝓑-coordinates. Verify `P_{𝓒 → 𝓑} · P_{𝓑 → 𝓒} = I`.

### E5 — Polynomial subspace with two conditions

Let `V = { p ∈ P₃ : p(0) = 0 and p(1) = 0 }` where `P₃` is polynomials of degree ≤ 3.

(a) Using the coefficient identification `p = a₀ + a₁ x + a₂ x² + a₃ x³ ↔ (a₀, a₁, a₂, a₃)`, write the defining equations as a linear system.

(b) Find a basis of `V` as polynomials. What is `dim V`?

---

### Solutions

### A1

In [ ]:
# All three axioms symbolically:
u1, u2, v1, v2, c = var('u1 u2 v1 v2 c')

def in_W(x, y):
    # z determined by 2x - y + 3z = 0  ⇒  z = (y - 2x)/3
    return vector(SR, [x, y, (y - 2*x)/3])

# Axiom 1
show('0 in W?', in_W(0, 0) == vector(SR, [0, 0, 0]))

# Axiom 2
u = in_W(u1, u2); v = in_W(v1, v2); s = u + v
show('sum residual:', (2*s[0] - s[1] + 3*s[2]).simplify_full())

# Axiom 3
cu = c * u
show('scalar residual:', (2*cu[0] - cu[1] + 3*cu[2]).simplify_full())

# Basis of W: parametrize (x, y) → (x, y, (y - 2x)/3).
# Take (x, y) = (1, 0) → (1, 0, -2/3); (x, y) = (0, 1) → (0, 1, 1/3).
b1 = vector(QQ, [1, 0, -2/3])
b2 = vector(QQ, [0, 1,  1/3])
B = column_matrix([b1, b2])
show('Basis (columns):'); show(B)
show('rank =', B.rank(), '  → dim W = 2')

### A2

In [ ]:
M = matrix(QQ, [[1, 2, 1, 0],
                [2, 4, 1, 1],
                [1, 2, 2, -1]])
show('M ='); show(M)
show('RREF(M) ='); show(M.rref())
show('pivots =', M.pivots())

basis_im = M.matrix_from_columns(list(M.pivots()))
show('Basis of im M:'); show(basis_im)
show('rank M =', M.rank())

# Kernel basis by hand recipe
free = [c for c in range(M.ncols()) if c not in M.pivots()]
Rref = M.rref()
basis_ker = []
for f in free:
    v = [0] * M.ncols()
    v[f] = 1
    for i, p in enumerate(M.pivots()):
        v[p] = -Rref[i, f]
    basis_ker.append(vector(QQ, v))
show('Basis of ker M:', basis_ker)
show('nullity =', len(basis_ker))
show('rank + nullity =', M.rank() + len(basis_ker), '  = n = ', M.ncols())

### A3

In [ ]:
b1 = vector(QQ, [1, 2, 0])
b2 = vector(QQ, [0, 1, 1])
b3 = vector(QQ, [1, 0, 1])
S = column_matrix([b1, b2, b3])
show('S ='); show(S)
show('det S =', S.det(), '  (non-zero → basis)')

x = vector(QQ, [3, 2, 1])
c = S.solve_right(x)
show('[x]_B =', c)
show('rebuild:', S * c)

### A4

In [ ]:
S_B = matrix(QQ, [[2, 1], [1, 3]])  # columns (2,1), (1,3)
S_C = matrix(QQ, [[1, 1], [0, 1]])  # columns (1,0), (1,1)

P_BtoC = S_C.inverse() * S_B
P_CtoB = S_B.inverse() * S_C

show('P_{B -> C} ='); show(P_BtoC)
show('P_{C -> B} ='); show(P_CtoB)
show('P_{C -> B} · P_{B -> C} ='); show(P_CtoB * P_BtoC)
# Should be the identity

### A5

In [ ]:
R.<x> = PolynomialRing(QQ)

# A polynomial a0 + a1 x + a2 x^2 + a3 x^3
# p(0) = 0 ⇒ a0 = 0
# p(1) = 0 ⇒ a0 + a1 + a2 + a3 = 0
#
# Combined with a0 = 0: a1 + a2 + a3 = 0, a0 = 0.
# Free variables: say a2 = s, a3 = t.  Then a1 = -s - t.
# So p = (-s - t) x + s x^2 + t x^3 = s (x^2 - x) + t (x^3 - x).

p1 = x^2 - x   # = x(x - 1)
p2 = x^3 - x   # = x(x^2 - 1) = x(x-1)(x+1)

show('p1(0) =', p1.subs(x=0), '  p1(1) =', p1.subs(x=1))
show('p2(0) =', p2.subs(x=0), '  p2(1) =', p2.subs(x=1))

def coeffs(p, deg=3):
    lst = p.list()
    return vector(QQ, lst + [0]*(deg+1 - len(lst)))

B = column_matrix([coeffs(p1), coeffs(p2)])
show('Basis matrix:'); show(B)
show('rank =', B.rank(), '  → dim V = 2')

---

## Where to next

- **Chapter 6 (Sage):** The polynomial space `P_n` and the matrix space `M_{m×n}` as full vector spaces. Linear maps between them — like differentiation `D: P_n → P_{n-1}` — get a matrix representation relative to a basis.
- **Chapter 7 (Sage):** Orthogonality — turn any basis into an *orthonormal* basis via Gram–Schmidt. That is QR factorization, and it makes coordinates trivially computable by dot products.
- **Chapter 9 (Sage):** Find bases in which a given map `T` has a *diagonal* matrix — the eigenvectors. Change of basis, applied.